<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/Phase50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 · Phase 50 — authorized real-MRI derivative source inventory and eligibility freeze

Phase 49 froze the currently executable synthetic evidence and made the comparator-role amendment explicit. The largest remaining scientific gap is no longer method construction; it is **external source evidence**.

Phase 50 searches the connected Drive for candidate neuroimaging derivative tables and evaluates whether the source evidence needed for the planned longitudinal MRI study actually exists.

It does **not** publish participant identifiers or row-level source data.

### Questions answered

1. Are candidate ADNI / FreeSurfer / longitudinal MRI derivative tables present?
2. Do they contain a stable person key and baseline / approximately 12-month time information?
3. Are bilateral hippocampal measurements identifiable?
4. Are entorhinal, middle-temporal, ventricular, and ICV-related features present?
5. Are measurement units explicit?
6. Are acquisition / study / series / image identifiers available?
7. Are extraction-model / software-version / provenance fields available?
8. Is clinical MCI status actually represented, rather than assumed from an earlier manuscript?
9. Has local authorized-use attestation been recorded for the candidate source?

### Privacy boundary

Public outputs contain only aggregate counts, Boolean field coverage, file-size metadata, and non-identifying SHA-derived file tokens. Exact Drive paths, RIDs/PTIDs, image identifiers, and row-level neuroimaging data remain private.


In [1]:
# CODE CELL 1/9 — Drive, Phase49 lock, helpers
import os, re, csv, json, gzip, hashlib, shutil, tempfile, zipfile, time, math
import datetime as dt
from pathlib import Path
from collections import Counter, defaultdict

PROJECT='ICHI2027_neuroimaging_fhir_omop_fidelity'
PHASE='50_real_mri_derivative_source_inventory_and_eligibility_freeze'
EXPECTED_P49_SHAREABLE='f10939f00827c2f4ef4b5d9fc7ea9f8002d8ced49f33775a170dd9d502f763f9'

MAX_PROFILE_BYTES=1_500_000_000
MAX_CANDIDATES_TO_PROFILE=12

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(2*1024*1024),b''):
            h.update(b)
    return h.hexdigest()

def token_for_path(p):
    return hashlib.sha256(str(Path(p).resolve()).encode('utf-8')).hexdigest()[:16]

def write_csv(path,rows,fieldnames=None):
    rows=list(rows)
    if fieldnames is None:
        if not rows: raise ValueError('fieldnames required for empty CSV')
        fieldnames=list(rows[0].keys())
    with Path(path).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=fieldnames)
        w.writeheader(); w.writerows(rows)

if os.environ.get('PHASE50_LOCAL_DRIVE_ROOT'):
    MYDRIVE=Path(os.environ['PHASE50_LOCAL_DRIVE_ROOT']).expanduser().resolve()
    ROOTS=[MYDRIVE]
    print('Offline Drive-mirror mode:',MYDRIVE)
else:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
    MYDRIVE=Path('/content/drive/MyDrive')
    assert MYDRIVE.is_dir()
    ROOTS=[MYDRIVE]
    sd=Path('/content/drive/Shareddrives')
    if sd.is_dir(): ROOTS.append(sd)

hint=os.environ.get('PHASE50_DRIVE_FOLDER_HINT','').strip()
if hint:
    hp=Path(hint).expanduser()
    if not hp.is_absolute(): hp=MYDRIVE/hp
    if not hp.is_dir(): raise FileNotFoundError(hp)
    ROOTS=[hp.resolve()]

WORK=Path(tempfile.mkdtemp(prefix='ichi2027_phase50_'))
STAGE=WORK/'outputs'; STAGE.mkdir()
OUT_BASE=Path(os.environ.get('PHASE50_OUTPUT_DIR',str(MYDRIVE/'ICHI2027'/'Phase50')))
OUT=OUT_BASE/dt.datetime.now(dt.timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')
PRIVATE=MYDRIVE/'ICHI2027'/'_PRIVATE_REAL_MRI'
PRIVATE.mkdir(parents=True,exist_ok=True)

print('Phase 50 output:',OUT)
print('Private source-evidence workspace prepared.')


Mounted at /content/drive
Phase 50 output: /content/drive/MyDrive/ICHI2027/Phase50/run_20261001T043051_504751Z
Private source-evidence workspace prepared.


In [2]:
# CODE CELL 2/9 — Verify exact Phase49 shareable evidence

def name_matches(actual,stem,ext):
    return re.fullmatch(re.escape(stem)+r'(?:-\d+|\s*\(\d+\))?'+re.escape(ext),actual,re.I) is not None

good=[]
for root in ROOTS:
    for here,dirs,files in os.walk(root,topdown=True,followlinks=False):
        dirs[:]=[d for d in dirs if d not in {'.git','node_modules','.ipynb_checkpoints'} and not d.startswith('.Trash')]
        for fn in files:
            if name_matches(fn,'ICHI2027_Phase49_SHAREABLE','.json'):
                p=Path(here)/fn
                try:
                    if sha256(p)==EXPECTED_P49_SHAREABLE:good.append(p)
                except OSError:pass

if not good:
    raise FileNotFoundError('Verified Phase49 SHAREABLE.json was not found in Drive.')

P49=sorted(good,key=str)[0]
p49=json.loads(P49.read_text('utf-8'))

assert p49['phase']=='49_comparator_role_amendment_and_executable_synthetic_results_freeze'
assert p49['armA_executed'] is False
assert p49['armC_neuroimaging_executed'] is False
assert p49['armD_developer_authored_synthetic_executed'] is True
assert p49['real_authorized_mri_processed'] is False
assert p49['independently_blinded_external_validation'] is False
assert p49['full_A_B_C_D_evaluation'] is False

print('PASS: exact Phase49 results freeze verified.')
print('Protocol amendment:',p49['protocol_amendment_sha256'])


PASS: exact Phase49 results freeze verified.
Protocol amendment: ab797f913e4afca4e2673aa6e2084bd7bce71cdc2c8a23b6af7b948df69827ed


In [3]:
# CODE CELL 3/9 — Private authorization-attestation template and validation

ATTEST=PRIVATE/'AUTHORIZED_SOURCE_ATTESTATION.json'
ATTEST_TEMPLATE=PRIVATE/'AUTHORIZED_SOURCE_ATTESTATION_TEMPLATE.json'

template={
 'source_family':'ADNI_or_other_authorized_neuroimaging_derivative',
 'authorized_access':False,
 'authorized_for_local_research_analysis':False,
 'data_use_terms_reviewed':False,
 'participant_level_public_release_prohibited_or_restricted':True,
 'attestation_date_utc':'YYYY-MM-DD',
 'attested_by':'',
 'source_description':'',
 'notes':'This is a local investigator attestation, not a substitute for the source data-use agreement.'
}
if not ATTEST_TEMPLATE.exists():
    ATTEST_TEMPLATE.write_text(json.dumps(template,indent=2)+'\n',encoding='utf-8')

attestation={}
attestation_valid=False
if ATTEST.exists():
    try:
        attestation=json.loads(ATTEST.read_text('utf-8'))
        day=dt.date.fromisoformat(str(attestation['attestation_date_utc']))
        attestation_valid=(
          day<=dt.datetime.now(dt.timezone.utc).date()
          and attestation.get('authorized_access') is True
          and attestation.get('authorized_for_local_research_analysis') is True
          and attestation.get('data_use_terms_reviewed') is True
          and bool(str(attestation.get('attested_by','')).strip())
          and bool(str(attestation.get('source_description','')).strip())
        )
    except Exception:
        attestation_valid=False

print('Local authorized-source attestation valid:',attestation_valid)
print('Template exists in private source-evidence workspace:',ATTEST_TEMPLATE.is_file())


Local authorized-source attestation valid: False
Template exists in private source-evidence workspace: True


In [4]:
# CODE CELL 4/9 — Header-only scan for candidate neuroimaging derivative tables

SUPPORTED={'.csv','.tsv','.txt','.gz','.parquet','.xlsx','.xls'}
EXCLUDE_NAME=re.compile(
    r'^(ICHI2027_|Phase\d+|phase\d+)|synthetic|fixture|holdout|challenge|shareable|'
    r'arm[_ -]?[abcd]|ablation|policy_actions|cohort_consequences|binding_review',
    re.I
)

FILENAME_HINTS={
 'adni':4,'freesurfer':5,'free_surfer':5,'ucsf':4,'mri':3,'neuro':2,'trace':2,
 'fsx':3,'cross':1,'long':1,'roi':2,'volume':2
}
HEADER_HINTS={
 'rid':4,'ptid':4,'subject':3,'participant':3,'viscode':4,'visit':2,'timepoint':3,
 'hippoc':5,'entorh':5,'middletemp':4,'middle_temporal':4,'ventric':3,
 'icv':4,'intracranial':4,'etiv':4,'imageuid':3,'studyuid':3,'seriesuid':3,
 'examdate':2,'scandate':2,'diagnosis':2,'dx':2
}

def normalize_col(c):
    return re.sub(r'[^a-z0-9]+','_',str(c).strip().lower()).strip('_')

def header_from_file(p):
    suf=p.suffix.lower()
    try:
        if suf=='.gz':
            with gzip.open(p,'rt',encoding='utf-8-sig',errors='replace',newline='') as f:
                first=f.readline()
                sep='\t' if first.count('\t')>first.count(',') else ','
                return [x.strip() for x in next(csv.reader([first],delimiter=sep))]
        if suf in {'.csv','.tsv','.txt'}:
            with p.open('r',encoding='utf-8-sig',errors='replace',newline='') as f:
                first=f.readline()
                sep='\t' if (suf=='.tsv' or first.count('\t')>first.count(',')) else ','
                return [x.strip() for x in next(csv.reader([first],delimiter=sep))]
        if suf=='.parquet':
            import pandas as pd
            return list(pd.read_parquet(p,engine='pyarrow').columns)
        if suf in {'.xlsx','.xls'}:
            import pandas as pd
            return list(pd.read_excel(p,nrows=0).columns)
    except Exception:
        return []
    return []

candidates=[]
private_paths={}
for root in ROOTS:
    for here,dirs,files in os.walk(root,topdown=True,followlinks=False):
        dirs[:]=[d for d in dirs if d not in {'.git','node_modules','.ipynb_checkpoints'} and not d.startswith('.Trash')]
        for fn in files:
            p=Path(here)/fn
            if p.suffix.lower() not in SUPPORTED:continue
            if EXCLUDE_NAME.search(fn):continue
            # Avoid generated project outputs as candidates.
            path_low=str(p).lower()
            if '/ichi2027/phase' in path_low or '\\ichi2027\\phase' in path_low:continue
            try:size=p.stat().st_size
            except OSError:continue
            if size<=0:continue
            cols=header_from_file(p)
            ncols=[normalize_col(c) for c in cols]
            blob=' '.join(ncols)
            score=0
            lowfn=fn.lower()
            for k,w in FILENAME_HINTS.items():
                if k in lowfn:score+=w
            for k,w in HEADER_HINTS.items():
                if k in blob:score+=w
            if score<5:continue
            tok=token_for_path(p)
            candidates.append({
              'file_token':tok,
              'extension':p.suffix.lower(),
              'bytes':size,
              'header_column_count':len(cols),
              'candidate_score':score,
              'contains_adni_filename_hint':'adni' in lowfn,
              'contains_freesurfer_filename_hint':('freesurfer' in lowfn or 'free_surfer' in lowfn or 'fsx' in lowfn or 'ucsf' in lowfn),
              'header_person_hint':any(x in blob for x in ['rid','ptid','subject','participant','person_id','patient_id']),
              'header_time_hint':any(x in blob for x in ['viscode','visit','timepoint','examdate','scandate']),
              'header_hippocampal_hint':'hippoc' in blob,
              'header_entorhinal_hint':'entorh' in blob,
              'header_icv_hint':any(x in blob for x in ['icv','intracranial','etiv']),
              'private_path_exported':False
            })
            private_paths[tok]=p

candidates=sorted(candidates,key=lambda r:(-r['candidate_score'],r['bytes']))
print('Candidate derivative tables found:',len(candidates))
for r in candidates[:10]:
    print(r['file_token'],'score',r['candidate_score'],'cols',r['header_column_count'],'bytes',r['bytes'])


Candidate derivative tables found: 11
069caf1537e082c6 score 27 cols 66 bytes 211294
fcfa1f7105fc6d0f score 6 cols 22 bytes 3502301
1a6ee54a4aae42f2 score 6 cols 22 bytes 3551371
fbfdd5f26c81ec45 score 6 cols 22 bytes 3551371
4b9b6fa45e69df6b score 6 cols 22 bytes 3551371
f5e00e13a807495b score 6 cols 22 bytes 3551376
e991fa12871b3947 score 6 cols 22 bytes 3551376
7e454f7593fdcbbd score 5 cols 12 bytes 1675
a06f3661f70e5a07 score 5 cols 12 bytes 1759
936affe78afbb3ea score 5 cols 11 bytes 34167


In [5]:
# CODE CELL 5/9 — Profile top candidate tables privately; export only aggregates

import pandas as pd
import numpy as np

PERSON_PATTERNS=['rid','ptid','subject_id','subject','participant_id','participant','person_id','patient_id']
VISIT_PATTERNS=['viscode2','viscode','visit_code','visit','timepoint','time_point']
DATE_PATTERNS=['examdate','scan_date','scandate','acquisition_date','study_date','date']
DX_PATTERNS=['dx','diagnosis','diagnostic','research_group','group']
UNIT_PATTERNS=['unit','units','ucum']
REGION_PATTERNS=['region','anatomy','structure','roi','feature_name']
LATERALITY_PATTERNS=['laterality','side','hemisphere']
VALUE_PATTERNS=['value','volume','measurement_value','numeric_value']
MODEL_PATTERNS=['algorithm_version','model_version','software_version','freesurfer_version','pipeline_version','version']
PROV_PATTERNS=['provenance','device','algorithm','software','processing','pipeline']
DICOM_PATTERNS=['studyuid','study_uid','seriesuid','series_uid','imageuid','image_uid','sopinstanceuid','dicom']

def choose_col(cols,patterns):
    norm={normalize_col(c):c for c in cols}
    for pat in patterns:
        if pat in norm:return norm[pat]
    for pat in patterns:
        for n,orig in norm.items():
            if pat in n:return orig
    return None

def load_table(p):
    suf=p.suffix.lower()
    if suf=='.gz':
        return pd.read_csv(p,compression='gzip',low_memory=False)
    if suf=='.csv':
        return pd.read_csv(p,low_memory=False)
    if suf=='.tsv':
        return pd.read_csv(p,sep='\t',low_memory=False)
    if suf=='.txt':
        try:return pd.read_csv(p,sep='\t',low_memory=False)
        except Exception:return pd.read_csv(p,low_memory=False)
    if suf=='.parquet':
        return pd.read_parquet(p)
    if suf in {'.xlsx','.xls'}:
        return pd.read_excel(p)
    raise ValueError(suf)

def side_from_name(name):
    n=normalize_col(name)
    if re.search(r'(^|_)(left|lh|l)($|_)',n):return 'L'
    if re.search(r'(^|_)(right|rh|r)($|_)',n):return 'R'
    return ''

def feature_from_name(name):
    n=normalize_col(name)
    if 'hippoc' in n:return 'hippocampus'
    if 'entorh' in n:return 'entorhinal'
    if 'middle_temporal' in n or 'middletemp' in n or 'midtemp' in n:return 'middle_temporal'
    if 'ventric' in n:return 'ventricle'
    if 'intracranial' in n or 'etiv' in n or re.search(r'(^|_)icv($|_)',n):return 'icv'
    return ''

def visit_class(v):
    s=str(v).strip().lower().replace(' ','').replace('_','').replace('-','')
    if s in {'bl','baseline','m0','m00','month0','screeningbaseline'}:return 'BASELINE'
    if s in {'m12','month12','12m','12month','12months','v12'}:return 'M12'
    m=re.search(r'(?:m|month|v)(\d+)',s)
    if m:
        try:
            mo=int(m.group(1))
            if mo==0:return 'BASELINE'
            if 9<=mo<=15:return 'M12'
        except:pass
    return ''

profile_rows=[]
field_rows=[]

for cand in candidates[:MAX_CANDIDATES_TO_PROFILE]:
    tok=cand['file_token']; p=private_paths[tok]
    if cand['bytes']>MAX_PROFILE_BYTES:
        profile_rows.append({
          'file_token':tok,'profile_status':'SKIPPED_TOO_LARGE','rows_profiled':0,
          'unique_persons':0,'baseline_persons':0,'m12_persons':0,'paired_baseline_m12_persons':0,
          'bilateral_hippocampus_columns':0,'entorhinal_columns':0,'middle_temporal_columns':0,
          'ventricle_columns':0,'icv_columns':0,'diagnosis_field_present':False,
          'mci_like_values_present':False,'dicom_or_image_id_field_present':False,
          'model_or_version_field_present':False,'explicit_unit_field_present':False,
          'private_row_level_exported':False
        })
        continue
    try:
        df=load_table(p)
    except Exception as e:
        profile_rows.append({
          'file_token':tok,'profile_status':'READ_ERROR','rows_profiled':0,
          'unique_persons':0,'baseline_persons':0,'m12_persons':0,'paired_baseline_m12_persons':0,
          'bilateral_hippocampus_columns':0,'entorhinal_columns':0,'middle_temporal_columns':0,
          'ventricle_columns':0,'icv_columns':0,'diagnosis_field_present':False,
          'mci_like_values_present':False,'dicom_or_image_id_field_present':False,
          'model_or_version_field_present':False,'explicit_unit_field_present':False,
          'private_row_level_exported':False
        })
        continue

    cols=list(df.columns)
    person=choose_col(cols,PERSON_PATTERNS)
    visit=choose_col(cols,VISIT_PATTERNS)
    datecol=choose_col(cols,DATE_PATTERNS)
    dx=choose_col(cols,DX_PATTERNS)
    unit=choose_col(cols,UNIT_PATTERNS)
    region=choose_col(cols,REGION_PATTERNS)
    lat=choose_col(cols,LATERALITY_PATTERNS)
    value=choose_col(cols,VALUE_PATTERNS)

    features=[(c,feature_from_name(c),side_from_name(c)) for c in cols]
    wide_hip=[x for x in features if x[1]=='hippocampus']
    hip_sides={x[2] for x in wide_hip if x[2]}
    ent=[x for x in features if x[1]=='entorhinal']
    mid=[x for x in features if x[1]=='middle_temporal']
    vent=[x for x in features if x[1]=='ventricle']
    icv=[x for x in features if x[1]=='icv']

    unique_persons=0; baseline_persons=0; m12_persons=0; paired=0
    if person:
        person_vals=df[person].dropna().astype(str)
        unique_persons=person_vals.nunique()
        if visit:
            vc=df[visit].map(visit_class)
            b=set(df.loc[vc=='BASELINE',person].dropna().astype(str))
            m=set(df.loc[vc=='M12',person].dropna().astype(str))
            baseline_persons=len(b); m12_persons=len(m); paired=len(b&m)
        elif datecol:
            tmp=df[[person,datecol]].dropna().copy()
            tmp['_date']=pd.to_datetime(tmp[datecol],errors='coerce')
            tmp=tmp.dropna(subset=['_date'])
            bset=set(); mset=set()
            for pid,g in tmp.groupby(person):
                ds=sorted(g['_date'].dt.normalize().unique())
                if not ds:continue
                bset.add(str(pid))
                first=pd.Timestamp(ds[0])
                if any(270 <= (pd.Timestamp(x)-first).days <= 455 for x in ds[1:]):
                    mset.add(str(pid))
            baseline_persons=len(bset); m12_persons=len(mset); paired=len(bset&mset)

    mci=False
    if dx:
        vals=' '.join(df[dx].dropna().astype(str).str.lower().unique()[:1000])
        mci=('mci' in vals or 'mild cognitive' in vals)

    normalized_cols=[normalize_col(c) for c in cols]
    dicom_present=any(any(pat in n for pat in DICOM_PATTERNS) for n in normalized_cols)
    model_present=any(any(pat in n for pat in MODEL_PATTERNS+PROV_PATTERNS) for n in normalized_cols)

    profile_rows.append({
      'file_token':tok,'profile_status':'PROFILED','rows_profiled':len(df),
      'unique_persons':unique_persons,'baseline_persons':baseline_persons,'m12_persons':m12_persons,
      'paired_baseline_m12_persons':paired,
      'bilateral_hippocampus_columns':int({'L','R'}<=hip_sides),
      'entorhinal_columns':len(ent),'middle_temporal_columns':len(mid),
      'ventricle_columns':len(vent),'icv_columns':len(icv),
      'diagnosis_field_present':bool(dx),'mci_like_values_present':mci,
      'dicom_or_image_id_field_present':dicom_present,
      'model_or_version_field_present':model_present,
      'explicit_unit_field_present':bool(unit),
      'private_row_level_exported':False
    })

    for dimension,present,detail in [
      ('person_key',bool(person),'present' if person else 'absent'),
      ('visit_or_date',bool(visit or datecol),'present' if (visit or datecol) else 'absent'),
      ('bilateral_hippocampus_wide_columns',{'L','R'}<=hip_sides,f'{len(wide_hip)} hippocampus-like columns'),
      ('entorhinal',bool(ent or (region and lat)),'wide or long-format evidence'),
      ('middle_temporal',bool(mid or (region and lat)),'wide or long-format evidence'),
      ('ventricular',bool(vent or region),'wide or long-format evidence'),
      ('icv',bool(icv),'wide feature evidence'),
      ('explicit_units',bool(unit),'explicit unit field' if unit else 'no explicit unit column detected'),
      ('dicom_or_image_link',dicom_present,'study/series/image-like identifier field scan'),
      ('model_or_provenance',model_present,'algorithm/software/device/processing field scan'),
      ('diagnostic_status',bool(dx),'diagnostic/group field scan'),
    ]:
        field_rows.append({
          'file_token':tok,'dimension':dimension,'present':bool(present),'detail':detail
        })

print('Candidate tables profiled:',sum(r['profile_status']=='PROFILED' for r in profile_rows))


Candidate tables profiled: 9


In [6]:
# CODE CELL 6/9 — Select the strongest source candidate and freeze readiness gates

profiled=[r for r in profile_rows if r['profile_status']=='PROFILED']
candidate_score_map={r['file_token']:r['candidate_score'] for r in candidates}

def readiness_score(r):
    return (
      min(int(r['unique_persons']>0),1)*3
      +min(int(r['paired_baseline_m12_persons']>0),1)*4
      +int(r['bilateral_hippocampus_columns'])*5
      +min(int(r['entorhinal_columns']>0),1)*2
      +min(int(r['icv_columns']>0),1)*2
      +int(r['dicom_or_image_id_field_present'])*2
      +int(r['model_or_version_field_present'])*2
      +int(r['explicit_unit_field_present'])*2
      +int(r['diagnosis_field_present'])*2
      +candidate_score_map.get(r['file_token'],0)/10
    )

selected=max(profiled,key=readiness_score) if profiled else None
selected_token=selected['file_token'] if selected else None

gates=[
 {
  'gate':'candidate_neuroimaging_derivative_table_found',
  'passed':selected is not None,
  'evidence':f'{len(profiled)} candidate tables profiled'
 },
 {
  'gate':'local_authorized_use_attestation_valid',
  'passed':attestation_valid,
  'evidence':'Private local attestation validated' if attestation_valid else 'No valid local attestation found'
 },
 {
  'gate':'stable_person_key_available',
  'passed':bool(selected and selected['unique_persons']>0),
  'evidence':f"{selected['unique_persons']} unique persons" if selected else 'No selected candidate'
 },
 {
  'gate':'baseline_and_12_month_pairing_available',
  'passed':bool(selected and selected['paired_baseline_m12_persons']>0),
  'evidence':f"{selected['paired_baseline_m12_persons']} paired persons" if selected else 'No selected candidate'
 },
 {
  'gate':'bilateral_hippocampal_measurements_identifiable',
  'passed':bool(selected and selected['bilateral_hippocampus_columns']),
  'evidence':'Both left and right hippocampus-like wide fields detected' if selected and selected['bilateral_hippocampus_columns'] else 'Not established'
 },
 {
  'gate':'explicit_units_available',
  'passed':bool(selected and selected['explicit_unit_field_present']),
  'evidence':'Explicit unit column detected' if selected and selected['explicit_unit_field_present'] else 'Units not explicitly established'
 },
 {
  'gate':'imaging_source_identifier_available',
  'passed':bool(selected and selected['dicom_or_image_id_field_present']),
  'evidence':'Study/series/image-like identifier field detected' if selected and selected['dicom_or_image_id_field_present'] else 'No real imaging-link field established'
 },
 {
  'gate':'extraction_model_or_provenance_available',
  'passed':bool(selected and selected['model_or_version_field_present']),
  'evidence':'Algorithm/software/device/processing field detected' if selected and selected['model_or_version_field_present'] else 'Not established'
 },
 {
  'gate':'clinical_mci_status_available',
  'passed':bool(selected and selected['diagnosis_field_present'] and selected['mci_like_values_present']),
  'evidence':'MCI-like diagnostic values detected' if selected and selected['diagnosis_field_present'] and selected['mci_like_values_present'] else 'MCI source status not established'
 }
]

passed=sum(int(g['passed']) for g in gates)
print('Selected private source token:',selected_token)
print('Real-source readiness gates:',passed,'/',len(gates))
for g in gates:
    print(('PASS' if g['passed'] else 'BLOCK'),g['gate'],'-',g['evidence'])


Selected private source token: 069caf1537e082c6
Real-source readiness gates: 2 / 9
PASS candidate_neuroimaging_derivative_table_found - 9 candidate tables profiled
BLOCK local_authorized_use_attestation_valid - No valid local attestation found
PASS stable_person_key_available - 258 unique persons
BLOCK baseline_and_12_month_pairing_available - 0 paired persons
BLOCK bilateral_hippocampal_measurements_identifiable - Not established
BLOCK explicit_units_available - Units not explicitly established
BLOCK imaging_source_identifier_available - No real imaging-link field established
BLOCK extraction_model_or_provenance_available - Not established
BLOCK clinical_mci_status_available - MCI source status not established


In [7]:
# CODE CELL 7/9 — Private longitudinal eligibility audit on selected candidate (aggregate output only)

eligibility={
 'selected_file_token':selected_token or '',
 'eligibility_status':'NOT_EVALUABLE',
 'source_rows_profiled':0,
 'source_unique_persons':0,
 'persons_with_baseline':0,
 'persons_with_approx_12m':0,
 'persons_with_baseline_and_approx_12m':0,
 'persons_with_bilateral_hippocampus_at_both_timepoints':0,
 'persons_with_source_mci_like_status_and_pairing':0,
 'source_ids_exported':False
}

if selected:
    p=private_paths[selected_token]
    df=load_table(p)
    cols=list(df.columns)
    person=choose_col(cols,PERSON_PATTERNS)
    visit=choose_col(cols,VISIT_PATTERNS)
    datecol=choose_col(cols,DATE_PATTERNS)
    dx=choose_col(cols,DX_PATTERNS)

    eligibility['source_rows_profiled']=len(df)
    eligibility['source_unique_persons']=selected['unique_persons']
    eligibility['persons_with_baseline']=selected['baseline_persons']
    eligibility['persons_with_approx_12m']=selected['m12_persons']
    eligibility['persons_with_baseline_and_approx_12m']=selected['paired_baseline_m12_persons']

    # Wide-table bilateral hippocampus audit only when left/right hippocampus columns are identifiable.
    hip_cols=[c for c in cols if feature_from_name(c)=='hippocampus']
    left=[c for c in hip_cols if side_from_name(c)=='L']
    right=[c for c in hip_cols if side_from_name(c)=='R']

    if person and left and right and (visit or datecol):
        temp=df.copy()
        if visit:
            temp['_tp']=temp[visit].map(visit_class)
        else:
            temp['_date']=pd.to_datetime(temp[datecol],errors='coerce')
            temp['_tp']=''
            for pid,ix in temp.dropna(subset=[person,'_date']).groupby(person).groups.items():
                dates=temp.loc[ix,'_date']
                first=dates.min()
                delta=(dates-first).dt.days
                temp.loc[ix[(delta<=30).to_numpy()],'_tp']='BASELINE'
                temp.loc[ix[((delta>=270)&(delta<=455)).to_numpy()],'_tp']='M12'

        temp['_hip_L']=temp[left].notna().any(axis=1)
        temp['_hip_R']=temp[right].notna().any(axis=1)
        good=temp[temp['_hip_L'] & temp['_hip_R'] & temp['_tp'].isin(['BASELINE','M12'])]
        bilateral=set()
        for pid,g in good.groupby(person):
            if {'BASELINE','M12'}<=set(g['_tp']):
                bilateral.add(str(pid))
        eligibility['persons_with_bilateral_hippocampus_at_both_timepoints']=len(bilateral)

        if dx:
            mci_people=set(
                temp.loc[
                    temp[dx].astype(str).str.lower().str.contains(r'\bmci\b|mild cognitive',regex=True,na=False),
                    person
                ].dropna().astype(str)
            )
            eligibility['persons_with_source_mci_like_status_and_pairing']=len(bilateral & mci_people)

    eligibility['eligibility_status']='AGGREGATE_SOURCE_AUDIT_COMPLETED'

print('Private aggregate eligibility audit:')
for k,v in eligibility.items():
    if k!='selected_file_token':print(k,':',v)


Private aggregate eligibility audit:
eligibility_status : AGGREGATE_SOURCE_AUDIT_COMPLETED
source_rows_profiled : 258
source_unique_persons : 258
persons_with_baseline : 0
persons_with_approx_12m : 0
persons_with_baseline_and_approx_12m : 0
persons_with_bilateral_hippocampus_at_both_timepoints : 0
persons_with_source_mci_like_status_and_pairing : 0
source_ids_exported : False


In [8]:
# CODE CELL 8/9 — Claim boundary and Phase51 decision

ready_for_real_method=all(g['passed'] for g in gates if g['gate'] in {
 'candidate_neuroimaging_derivative_table_found',
 'local_authorized_use_attestation_valid',
 'stable_person_key_available',
 'baseline_and_12_month_pairing_available',
 'bilateral_hippocampal_measurements_identifiable'
})

ready_for_official_armA=all(g['passed'] for g in gates if g['gate'] in {
 'candidate_neuroimaging_derivative_table_found',
 'local_authorized_use_attestation_valid',
 'stable_person_key_available',
 'baseline_and_12_month_pairing_available',
 'bilateral_hippocampal_measurements_identifiable',
 'explicit_units_available',
 'imaging_source_identifier_available'
})

if ready_for_real_method and selected:
    next_step='PHASE51_FREEZE_REAL_SOURCE_MANIFEST_AND_RUN_SOURCE_SIDE_NEUROIMAGING_METHOD'
elif selected and not attestation_valid:
    next_step='BLOCKED_PENDING_LOCAL_AUTHORIZED_USE_ATTESTATION'
elif selected:
    next_step='BLOCKED_SOURCE_FIELDS_INCOMPLETE_REVIEW_PRIVATE_CANDIDATE'
else:
    next_step='BLOCKED_NO_REAL_NEUROIMAGING_DERIVATIVE_CANDIDATE_FOUND'

claim_boundary=[
 {'claim':'Candidate real neuroimaging derivative data exist','permitted':selected is not None,
  'note':'File candidacy is based on private filename/header/content profiling, not clinical validation.'},
 {'claim':'Source is authorized for local research analysis','permitted':attestation_valid,
  'note':'Requires valid private local investigator attestation; notebook does not infer authorization from filename.'},
 {'claim':'Real MRI cohort has been analyzed by the proposed method','permitted':False,
  'note':'Phase50 inventories source evidence only; proposed method is not run on real data here.'},
 {'claim':'Official OMOP/MI-CDM Arm A has executed','permitted':False,
  'note':'Vocabulary, source Procedure/UID, and official target-load gates remain separate.'},
 {'claim':'Clinical MCI cohort is established','permitted':bool(selected and selected['diagnosis_field_present'] and selected['mci_like_values_present']),
  'note':'Only permitted when diagnostic source values are actually present in the selected source table.'}
]

print('Ready for next real-source method phase:',ready_for_real_method)
print('Ready for official Arm A source side:',ready_for_official_armA)
print('Next step:',next_step)


Ready for next real-source method phase: False
Ready for official Arm A source side: False
Next step: BLOCKED_PENDING_LOCAL_AUTHORIZED_USE_ATTESTATION


In [9]:
# CODE CELL 9/9 — Public-safe outputs and Drive read-back verification

def save(name,rows,fields=None):
    p=STAGE/name
    write_csv(p,rows,fields)
    return p

public_candidates=[]
for r in candidates:
    public_candidates.append(dict(r))

artifacts=[]
artifacts.append(save('ICHI2027_Phase50_SOURCE_CANDIDATE_INVENTORY.csv',public_candidates if public_candidates else [{
 'file_token':'','extension':'','bytes':0,'header_column_count':0,'candidate_score':0,
 'contains_adni_filename_hint':False,'contains_freesurfer_filename_hint':False,
 'header_person_hint':False,'header_time_hint':False,'header_hippocampal_hint':False,
 'header_entorhinal_hint':False,'header_icv_hint':False,'private_path_exported':False
}]))
artifacts.append(save('ICHI2027_Phase50_SOURCE_PROFILE_SUMMARY.csv',profile_rows if profile_rows else [{
 'file_token':'','profile_status':'NO_CANDIDATE','rows_profiled':0,'unique_persons':0,
 'baseline_persons':0,'m12_persons':0,'paired_baseline_m12_persons':0,
 'bilateral_hippocampus_columns':0,'entorhinal_columns':0,'middle_temporal_columns':0,
 'ventricle_columns':0,'icv_columns':0,'diagnosis_field_present':False,
 'mci_like_values_present':False,'dicom_or_image_id_field_present':False,
 'model_or_version_field_present':False,'explicit_unit_field_present':False,
 'private_row_level_exported':False
}]))
artifacts.append(save('ICHI2027_Phase50_FIELD_COVERAGE.csv',field_rows if field_rows else [{
 'file_token':'','dimension':'NO_CANDIDATE','present':False,'detail':'No candidate source profiled'
}]))
artifacts.append(save('ICHI2027_Phase50_REAL_SOURCE_READINESS_GATES.csv',gates))
artifacts.append(save('ICHI2027_Phase50_AGGREGATE_ELIGIBILITY.csv',[eligibility]))
artifacts.append(save('ICHI2027_Phase50_CLAIM_BOUNDARY.csv',claim_boundary))

readme=STAGE/'ICHI2027_Phase50_README.txt'
readme.write_text(
 'Phase 50 performs a private source-evidence inventory for real neuroimaging derivative tables.\n'
 'Public outputs contain no RIDs, PTIDs, image IDs, row-level source data, or private Drive paths.\n'
 'A file is not called authorized merely because it appears to be ADNI/FreeSurfer; authorization requires the private local attestation.\n'
 'Phase 50 does not execute the proposed neuroimaging method on real data and does not execute official OMOP/MI-CDM Arm A.\n'
 'The selected source token is a SHA-derived non-identifying path token used only to link public aggregate evidence across outputs.\n',
 encoding='utf-8'
)
artifacts.append(readme)

summary={
 'project':PROJECT,
 'phase':PHASE,
 'run_utc':dt.datetime.now(dt.timezone.utc).isoformat(),
 'input_phase49_shareable_sha256':sha256(P49),
 'privacy':'PUBLIC_AGGREGATES_ONLY_NO_SOURCE_IDS_NO_PRIVATE_PATHS',
 'candidate_source_tables_found':len(candidates),
 'candidate_source_tables_profiled':sum(r['profile_status']=='PROFILED' for r in profile_rows),
 'selected_source_token':selected_token,
 'selected_source_profile':selected if selected else None,
 'authorization_attestation_valid':attestation_valid,
 'readiness_gates_passed':passed,
 'readiness_gates_total':len(gates),
 'aggregate_eligibility':eligibility,
 'ready_for_real_source_method_phase':ready_for_real_method,
 'ready_for_official_armA_source_side':ready_for_official_armA,
 'next_step':next_step,
 'real_authorized_mri_processed_by_proposed_method':False,
 'official_omop_etl_executed':False,
 'official_micdm_load_executed':False,
 'independently_blinded_external_validation':False,
 'artifact_sha256':{p.name:sha256(p) for p in artifacts},
 'warning':'Candidate-source profiling and aggregate eligibility auditing do not establish authorization, clinical MCI validity, official target transformation, or real-data method performance.'
}
summary_path=STAGE/'ICHI2027_Phase50_SHAREABLE.json'
summary_path.write_text(json.dumps(summary,sort_keys=True,indent=2)+'\n',encoding='utf-8')
artifacts.append(summary_path)

zip_path=STAGE/'ICHI2027_Phase50_PUBLIC_OUTPUTS.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in artifacts:z.write(p,p.name)

OUT.mkdir(parents=True,exist_ok=False)
for p in artifacts+[zip_path]:
    dest=OUT/p.name
    for attempt in range(3):
        try:
            shutil.copyfile(p,dest)
            assert sha256(dest)==sha256(p)
            break
        except Exception as e:
            if attempt==2:raise RuntimeError('Drive save/readback verification failed for '+p.name) from e
            time.sleep(1+attempt)

print('SUCCESS: Phase 50 source-evidence inventory completed.')
print('Candidates found:',len(candidates))
print('Readiness gates:',passed,'/',len(gates))
print('Next step:',next_step)
print('Shareable:',OUT/summary_path.name)
print('Public ZIP:',OUT/zip_path.name)


SUCCESS: Phase 50 source-evidence inventory completed.
Candidates found: 11
Readiness gates: 2 / 9
Next step: BLOCKED_PENDING_LOCAL_AUTHORIZED_USE_ATTESTATION
Shareable: /content/drive/MyDrive/ICHI2027/Phase50/run_20261001T043051_504751Z/ICHI2027_Phase50_SHAREABLE.json
Public ZIP: /content/drive/MyDrive/ICHI2027/Phase50/run_20261001T043051_504751Z/ICHI2027_Phase50_PUBLIC_OUTPUTS.zip


## Interpretation

Phase 50 is intentionally a gate, not a forced real-data experiment.

If an authorized source with person linkage, longitudinal pairing, and bilateral hippocampal evidence is verified, the next notebook can freeze a private source manifest and run the already-frozen neuroimaging method on that real derivative evidence without retuning.

If those fields are missing, the correct result is a source-data limitation. The notebook should not manufacture clinical procedures, DICOM UIDs, units, diagnostic status, or acquisition provenance merely to make the planned experiment executable.
